# Projet FinanceCore : Ingénierie des Données Bancaires
## Étape 1 : Importation des bibliothèques et Connexion

In [2]:
import os
import pandas as pd
from sqlalchemy import create_engine, text, MetaData, Table
from sqlalchemy.dialects.postgresql import insert
from dotenv import load_dotenv

load_dotenv()

engine_url = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
engine = create_engine(engine_url)

## Étape 2 : Création du Schéma Relationnel (3NF)

In [3]:
create_tables_sql = """
DROP VIEW IF EXISTS vue_transactions_details CASCADE;
DROP TABLE IF EXISTS Transaction CASCADE;
DROP TABLE IF EXISTS Compte CASCADE;
DROP TABLE IF EXISTS Client CASCADE;
DROP TABLE IF EXISTS Agence CASCADE;
DROP TABLE IF EXISTS Produit CASCADE;
DROP TABLE IF EXISTS Categorie CASCADE;

CREATE TABLE Agence (
    agence_id SERIAL PRIMARY KEY,
    nom_agence VARCHAR(100) UNIQUE NOT NULL
);

CREATE TABLE Produit (
    produit_id SERIAL PRIMARY KEY,
    nom_produit VARCHAR(100) UNIQUE NOT NULL
);

CREATE TABLE Categorie (
    categorie_id SERIAL PRIMARY KEY,
    nom_categorie VARCHAR(100) UNIQUE NOT NULL
);

CREATE TABLE Client (
    client_id VARCHAR(20) PRIMARY KEY,
    score_credit_client NUMERIC,
    nom_segment VARCHAR(50)
);

CREATE TABLE Compte (
    compte_id SERIAL PRIMARY KEY,
    client_id VARCHAR(20) REFERENCES Client(client_id) ON DELETE CASCADE,
    agence_id INT REFERENCES Agence(agence_id),
    produit_id INT REFERENCES Produit(produit_id),
    categorie_risque VARCHAR(50),
    UNIQUE (client_id, agence_id, produit_id)
);

CREATE TABLE Transaction (
    transaction_id VARCHAR(20) PRIMARY KEY,
    compte_id INT REFERENCES Compte(compte_id) ON DELETE CASCADE,
    categorie_id INT REFERENCES Categorie(categorie_id),
    date_transaction TIMESTAMP NOT NULL,
    montant NUMERIC NOT NULL,
    montant_eur NUMERIC NOT NULL,
    code_devise VARCHAR(3),
    type_operation VARCHAR(20),
    statut VARCHAR(20),
    solde_avant NUMERIC,
    is_anomalie BOOLEAN
);
"""
with engine.begin() as conn:
    conn.execute(text(create_tables_sql))

## Étape 3 : Utilitaires ETL et Chargement du CSV

In [4]:
def insert_on_conflict_nothing(table_name, engine, df):
    if df.empty: return
    metadata = MetaData()
    table = Table(table_name, metadata, autoload_with=engine)
    records = df.to_dict(orient='records')
    stmt = insert(table).values(records)
    do_nothing_stmt = stmt.on_conflict_do_nothing()
    with engine.begin() as conn:
        conn.execute(do_nothing_stmt)

## 3: Charger les données

In [5]:
# Cell 3: Charger les données
import pandas as pd
csv_path = '../data/financecore_clean.csv'

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
else:
    df = pd.read_csv('financecore_clean.csv')

df.head()

,transaction_id,client_id,date_transaction,montant,devise,taux_change_eur,montant_eur,categorie,produit,agence,...,semaine-jour,montant_eur_verifie,categorie_risque,total_credit,total_debit,solde_net,nb_transaction,montant_moyen,nb_produit,taux_rejet
0,TXN000559,CLI0060,2022-04-19 02:31:00,2050.42,EUR,1.00,2050.42,Depot especes,Compte Epargne,Marseille-Vieux-Port,...,1,2050.420000,Medium,18004.80,-3594.34,21599.14,12,1200.871667,7,5.572755
1,TXN001154,CLI0057,2024-06-20 20:51:00,-123.66,GBP,0.86,-143.79,Retrait DAB,Credit Consommation,Bordeaux-Meriadeck,...,3,-143.790698,High,1791.74,-5203.97,6995.71,9,-379.136667,7,4.589372
2,TXN000764,CLI0015,2024-08-28 05:03:00,-396.17,EUR,1.00,-396.17,Prelevement,PEA,Lyon-Part-Dieu,...,2,-396.170000,Medium,3752.75,-7055.30,10808.05,11,-300.231818,5,4.013378
3,TXN001598,CLI0045,2024-01-07 08:16:00,225.20,EUR,1.00,225.20,Paiement CB,Credit Consommation,Bordeaux-Meriadeck,...,6,225.200000,Low,6754.81,-17480.49,24235.30,17,-630.922353,7,4.589372
4,TXN001873,CLI0034,2024-08-11 19:52:00,935.32,EUR,1.00,935.32,Interets,Credit Immobilier,Bordeaux-Meriadeck,...,6,935.320000,High,5899.91,-5556.31,11456.22,13,26.430769,6,4.589372


## Étape 4 : Insertion des Dimensions et des Comptes

In [6]:
# 1. Agence, Produit, Categorie
load_agences = df[['agence']].drop_duplicates().rename(columns={'agence': 'nom_agence'})
insert_on_conflict_nothing('agence', engine, load_agences)

load_produits = df[['produit']].drop_duplicates().rename(columns={'produit': 'nom_produit'})
insert_on_conflict_nothing('produit', engine, load_produits)

load_categories = df[['categorie']].drop_duplicates().rename(columns={'categorie': 'nom_categorie'})
insert_on_conflict_nothing('categorie', engine, load_categories)

# 2. Client (Segment merged here)
load_clients = df[['client_id', 'score_credit_client', 'segment_client']].drop_duplicates(subset=['client_id'])
load_clients = load_clients.rename(columns={'segment_client': 'nom_segment'})
insert_on_conflict_nothing('client', engine, load_clients)

# 3. Compte (Mapping ids)
db_agences = pd.read_sql_table('agence', engine)
db_produits = pd.read_sql_table('produit', engine)

df_comptes = df[['client_id', 'agence', 'produit', 'categorie_risque']].drop_duplicates()
df_comptes = df_comptes.merge(db_agences, left_on='agence', right_on='nom_agence').merge(db_produits, left_on='produit', right_on='nom_produit')
insert_on_conflict_nothing('compte', engine, df_comptes[['client_id', 'agence_id', 'produit_id', 'categorie_risque']])

## Étape 5 : Chargement de la Table des Faits (Transactions)

In [7]:
db_comptes = pd.read_sql_table('compte', engine)
db_categories = pd.read_sql_table('categorie', engine)

# Mapping labels to IDs
df_trans = df.copy()
df_trans = df_trans.merge(db_agences, left_on='agence', right_on='nom_agence')
df_trans = df_trans.merge(db_produits, left_on='produit', right_on='nom_produit')
df_trans = df_trans.merge(db_categories, left_on='categorie', right_on='nom_categorie')

# Link to unique Compte
df_trans = df_trans.merge(db_comptes, on=['client_id', 'agence_id', 'produit_id'])

# Final insertion
transactions = df_trans[[
    'transaction_id', 'compte_id', 'categorie_id', 
    'date_transaction', 'montant', 'montant_eur', 
    'devise', 'type_operation', 'statut', 'solde_avant', 'is_anomalie'
]].rename(columns={'devise': 'code_devise'})

insert_on_conflict_nothing('transaction', engine, transactions)

## Étape 6 : Vérification de l'Intégrité

In [8]:
with engine.connect() as conn:
    nb_clients = conn.execute(text("SELECT COUNT(*) FROM Client")).scalar()
    nb_comptes = conn.execute(text("SELECT COUNT(*) FROM Compte")).scalar()
    nb_transactions = conn.execute(text("SELECT COUNT(*) FROM Transaction")).scalar()

print(f"Total Clients : {nb_clients}")
print(f"Total Comptes : {nb_comptes}")
print(f"Total Transactions : {nb_transactions}")

if nb_transactions == len(df):
    print("Intégrité validée : Toutes les transactions ont été insérées.")

Total Clients : 150
Total Comptes : 1012
Total Transactions : 2000
Intégrité validée : Toutes les transactions ont été insérées.


## Étape 7 : Analyses SQL Avancées

In [9]:
query_anomalie = """
SELECT cp.categorie_risque, COUNT(t.transaction_id) as total_transactions,
       SUM(CASE WHEN t.is_anomalie = TRUE THEN 1 ELSE 0 END) as nb_anomalies,
       ROUND(SUM(CASE WHEN t.is_anomalie = TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(t.transaction_id), 2) AS taux_anomalie_pct
FROM Compte cp
JOIN Transaction t ON cp.compte_id = t.compte_id
GROUP BY cp.categorie_risque
ORDER BY taux_anomalie_pct DESC;
"""
display(pd.read_sql_query(text(query_anomalie), engine))

,categorie_risque,total_transactions,nb_anomalies,taux_anomalie_pct
0,High,392,223,56.89
1,Low,479,27,5.64
2,Medium,1129,63,5.58
